# Lab 5 part 1
## Task 1

In [2]:
import pandas as pd

rentals_raw = pd.read_csv("../data/bike_rentals.csv")
rentals = rentals_raw.copy()

print(rentals.shape)
print(rentals.dtypes)

(403, 15)
rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object


In [3]:
column_empty_status = rentals.isna().all()
empty_columns_after = rentals.columns[column_empty_status]
print(column_empty_status)
print("Empty columns", empty_columns_after)

rental_id        False
station          False
bike_type        False
helmets_text     False
is_member        False
insured          False
paid_online      False
status           False
coupon_code       True
damage_notes      True
late_fee_note     True
start_date       False
end_date         False
price            False
rating           False
dtype: bool
Empty columns Index(['coupon_code', 'damage_notes', 'late_fee_note'], dtype='str')


In [4]:
rentals = rentals.dropna(axis="columns", how="all")
print(rentals.shape)
column_empty_status = rentals.isna().all()
empty_columns_after = rentals.columns[column_empty_status]
print(column_empty_status)
print("Empty columns after", empty_columns_after)

(403, 12)
rental_id       False
station         False
bike_type       False
helmets_text    False
is_member       False
insured         False
paid_online     False
status          False
start_date      False
end_date        False
price           False
rating          False
dtype: bool
Empty columns after Index([], dtype='str')


## Task 2

In [5]:
rentals["price"].head(3)

0      SEK 855.50
1    SEK 1,311.50
2      SEK 350.00
Name: price, dtype: str

In [6]:
# Step 1: remove the currency text and any remaning spaces
rentals["price"] = rentals["price"].str.replace("SEK", "").str.strip()
rentals["price"].head(3)

0      855.50
1    1,311.50
2      350.00
Name: price, dtype: str

In [7]:
# Step 2: remove the comma
rentals["price"] = rentals["price"].str.replace(",", "")
rentals["price"].head(3)

0     855.50
1    1311.50
2     350.00
Name: price, dtype: str

In [8]:
rentals["price"] = rentals["price"].astype(float)
print(rentals["price"].dtype)
print(rentals["price"].describe())

float64
count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64


In [9]:
top_3_price = rentals.sort_values("price", ascending=False)[["rental_id", "bike_type", "status", "price"]].head(3)
top_3_price

,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


## Task 3

In [10]:
missing_prices = rentals[rentals["price"].isna()]
print("Missing prices count", missing_prices.shape[0])
print("Missing price percentage", round(missing_prices.shape[0] / rentals.shape[0] * 100, 1))

Missing prices count 51
Missing price percentage 12.7


In [11]:
missing = rentals["price"].isna()
cancelled = rentals["status"] == "Cancelled"
pd.crosstab(missing, cancelled, rownames=["Price is missing"], colnames=["Rental cancelled"])

Rental cancelled,False,True
Price is missing,,
False,352,0
True,4,47


47 of the missing prices occur when the rental was cancelled, so they have a reason to be missing.  
The remaining 4 missing prices are missing due to some other reason.  

I will choose to keep the missing prices as they are, and not remove them or fill them in.  

## Task 4

In [12]:
tf_columns = [col for col in rentals.columns if set(rentals[col].dropna().unique()).issubset({"t", "f"})]
tf_columns

['is_member', 'insured', 'paid_online']

The column `status` is not in the list of columns, since its values are not only `t` or `f`.

In [13]:
for column in tf_columns:
    rentals[column] = rentals[column].map({"t": True, "f": False})

In [14]:
rentals.head()

,rental_id,station,bike_type,helmets_text,is_member,insured,paid_online,status,start_date,end_date,price,rating
0,R-5020,Södermalm,E-bike,1 helmet,False,True,True,Completed,2026-04-27,2026-04-29,855.5,5.0
1,R-5100,Södermalm,E-bike,1 helmet,False,True,True,Completed,2026-09-16,2026-09-19,1311.5,4.0
2,R-5055,Södermalm,City bike,3 helmets,False,False,True,Completed,2026-08-04,2026-08-06,350.0,NaN
3,R-5200,Södermalm,City bike,2 helmets,False,False,True,No-show,2026-05-30,2026-06-02,569.5,NaN
4,R-5109,Östermalm,City bike,2 helmets,False,True,False,No-show,2026-04-25,2026-04-29,828.0,NaN


In [15]:
for column in tf_columns:
    print(f"{column} has {rentals[column].isna().sum()} missing values")

is_member has 0 missing values
insured has 3 missing values
paid_online has 0 missing values


In [16]:
rentals.groupby("is_member")["price"].agg(["median", "count"])

,median,count
is_member,,
False,756.25,248
True,558.00,104


## Task 5

In [17]:
missing_values_after_start_date = rentals["start_date"].isna().sum()
missing_values_after_end_date = rentals["end_date"].isna().sum()

print("Missing values before for start date", missing_values_after_start_date)
print("Missing values before for end date", missing_values_after_end_date)

Missing values before for start date 0
Missing values before for end date 0


In [18]:
date_columns = ["start_date", "end_date"]
for column in date_columns:
    rentals[column] = pd.to_datetime(rentals[column], errors="coerce")

rentals.dtypes

rental_id                  str
station                    str
bike_type                  str
helmets_text               str
is_member                 bool
insured                 object
paid_online               bool
status                     str
start_date      datetime64[us]
end_date        datetime64[us]
price                  float64
rating                 float64
dtype: object

In [19]:
rentals.head()

,rental_id,station,bike_type,helmets_text,is_member,insured,paid_online,status,start_date,end_date,price,rating
0,R-5020,Södermalm,E-bike,1 helmet,False,True,True,Completed,2026-04-27,2026-04-29,855.5,5.0
1,R-5100,Södermalm,E-bike,1 helmet,False,True,True,Completed,2026-09-16,2026-09-19,1311.5,4.0
2,R-5055,Södermalm,City bike,3 helmets,False,False,True,Completed,2026-08-04,2026-08-06,350.0,NaN
3,R-5200,Södermalm,City bike,2 helmets,False,False,True,No-show,2026-05-30,2026-06-02,569.5,NaN
4,R-5109,Östermalm,City bike,2 helmets,False,True,False,No-show,2026-04-25,2026-04-29,828.0,NaN


In [20]:
missing_values_after_start_date = rentals["start_date"].isna().sum()
missing_values_after_end_date = rentals["end_date"].isna().sum()

print("Missing values after for start date", missing_values_after_start_date)
print("Missing values after for end date", missing_values_after_end_date)

Missing values after for start date 2
Missing values after for end date 0


In [21]:
rentals["rental_days"] = (rentals["end_date"] - rentals["start_date"]).dt.days
rentals["rental_days"].describe()

count    401.000000
mean       2.955112
std        1.835206
min       -2.000000
25%        1.000000
50%        3.000000
75%        4.000000
max        7.000000
Name: rental_days, dtype: float64

Something looks wrong for the min value, it should not take on negative values.

## Task 6

In [22]:
missing_ratings = rentals["rating"].isna().sum()
print("Missing ratings", missing_ratings)

Missing ratings 115


In [23]:
pd.crosstab(rentals["rating"].isna(), rentals["status"])

status,Cancelled,Completed,No-show
rating,,,
False,0,288,0
True,47,46,22


Not all the missing raings belong to cancelled rentals. Some of them are completed, and some of them have the status no-show.

In [24]:
rentals["has_rating"] = ~rentals["rating"].isna()
print(rentals["has_rating"].count())
print(rentals["has_rating"].value_counts())

403
has_rating
True     288
False    115
Name: count, dtype: int64


In [25]:
print("Rating median", rentals["rating"].median())
rentals["rating"].describe()

Rating median 5.0


count    288.000000
mean       4.225694
std        1.019547
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64

## Task 7

In [26]:
rentals["helmets_text"].value_counts(dropna=False)

helmets_text
1 helmet          146
2 helmets         127
No helmets         59
1 child helmet     47
3 helmets          20
NaN                 4
Name: count, dtype: int64

In [27]:
helmet_text_lowercase = rentals["helmets_text"].str.lower()
rentals["child_helmet"] = helmet_text_lowercase.str.contains("child")
print("Child helmets", rentals["child_helmet"].sum())

Child helmets 47


In [57]:
helmet_text_without_child_helmets = helmet_text_lowercase[~helmet_text_lowercase.str.contains("child")]
helmet_counts = helmet_text_without_child_helmets.str.extract(r"([\d]+)", expand=False)

print(helmet_counts.value_counts())
print("No number", helmet_counts.isna().sum())
print("Helmet count unique values", helmet_counts.unique())

helmets_text
1    146
2    127
3     20
Name: count, dtype: int64
No number 63
Helmet count unique values <StringArray>
['1', '3', '2', nan]
Length: 4, dtype: str


In [58]:
helmet_counts_with_zero = helmet_counts.where(~helmet_text_lowercase.str.contains("no helmets"), "0")

print(helmet_counts_with_zero.value_counts())
print("No number", helmet_counts_with_zero.isna().sum())
print("Helmet count unique values", helmet_counts_with_zero.unique())

helmets_text
1    146
2    127
0     59
3     20
Name: count, dtype: int64
No number 4
Helmet count unique values <StringArray>
['1', '3', '2', '0', nan]
Length: 5, dtype: str


In [61]:
rentals["helmets_count"] = helmet_counts_with_zero
rentals["helmets_count"].value_counts()

helmets_count
1    146
2    127
0     59
3     20
Name: count, dtype: int64